In [ ]:
import json                              # JSON serialization/deserialization
import sys                               # Python interpreter utilities

from pathlib import Path                 # Filesystem path handling
from datetime import datetime, timezone  # Date, time, and UTC timezone support

import websocket                         # WebSocket client communication

"""
=========================================================================================================
                                      MEXC WEBSOCKET ENDPOINTS
=========================================================================================================

┌──────────────────────────────────────────────────────────────┬───────────────────────────────────────────────┐
│ Endpoint                                                     │ Description                                   │
├──────────────────────────────────────────────────────────────┼───────────────────────────────────────────────┤
│ spot@public.limit.depth.v3.api.pb@{SYMBOL}@{LEVEL}           │ ORDER BOOK SNAPSHOT                           │
│                                                              │ • Full order book at the requested depth      │
│                                                              │ • Used for initial synchronization            │
├──────────────────────────────────────────────────────────────┼───────────────────────────────────────────────┤
│ spot@public.aggre.depth.v3.api.pb@100ms@{SYMBOL}             │ DELTA UPDATES (100 ms)                        │
│                                                              │ • Sends only order book changes               │
│                                                              │ • Update interval: 100 ms                     │
├──────────────────────────────────────────────────────────────┼───────────────────────────────────────────────┤
│ spot@public.aggre.depth.v3.api.pb@10ms@{SYMBOL}              │ HIGH-FREQUENCY DELTA (10 ms)                  │
│                                                              │ • Sends only order book changes               │
│                                                              │ • Update interval: 10 ms                      │
│                                                              │ • Ideal for liquidity analysis and HFT        │
├──────────────────────────────────────────────────────────────┼───────────────────────────────────────────────┤
│ spot@public.aggre.bookTicker.v3.api.pb@100ms@{SYMBOL}        │ BEST BID / ASK (100 ms)                       │
│                                                              │ Returns only:                                 │
│                                                              │ • Best Bid Price                              │
│                                                              │ • Best Ask Price                              │
│                                                              │ • Bid Quantity                                │
│                                                              │ • Ask Quantity                                │
├──────────────────────────────────────────────────────────────┼───────────────────────────────────────────────┤
│ spot@public.aggre.bookTicker.v3.api.pb@10ms@{SYMBOL}         │ FAST BEST BID / ASK (10 ms)                   │
│                                                              │ • Same fields as BookTicker                   │
│                                                              │ • Update interval: 10 ms                      │
│                                                              │ • Lowest-latency top-of-book feed             │
└──────────────────────────────────────────────────────────────┴───────────────────────────────────────────────┘

Legend
------
Snapshot      = Complete state of the order book.
Delta Update  = Only changes since the previous update.
BookTicker    = Top of book (best bid/ask) only.
"""

# Define the filesystem path containing the generated Protocol Buffer
# Python modules (.proto compiled to *_pb2.py files). 
PROTO_PATH = Path(
    r"D:\Data Projects\MEXC API Architecture\websocket-proto"
)

# Add the Protocol Buffer directory to Python's module search path
# so custom modules can be imported at runtime.
sys.path.append(str(PROTO_PATH))

# Import the generated Protocol Buffer message wrapper used to
# deserialize binary WebSocket messages received from the MEXC API.
from PushDataV3ApiWrapper_pb2 import PushDataV3ApiWrapper

# Secure WebSocket endpoint for connecting to the MEXC public
# market data streaming service.
WS_URL = "wss://wbs-api.mexc.com/ws"

# Trading pair to subscribe to.
# Format: <Base Asset><Quote Asset>
# Example: ETHUSDT = Ethereum priced in Tether (USDT).
SYMBOL = "ETHUSDT"

# Construct the WebSocket subscription topic.
#
# Topic format:
# spot@public.aggre.depth.v3.api.pb@10ms@<SYMBOL>
#
# Components:
#   spot        - Spot market
#   public      - Public data stream (no authentication required)
#   aggre.depth - Aggregated order book depth updates
#   v3          - Version 3 API
#   api.pb      - Protocol Buffer encoded messages
#   10ms        - Update interval (approximately every 10 ms)
#   SYMBOL      - Trading pair (e.g., ETHUSDT)
ENDPOINT = f"spot@public.aggre.depth.v3.api.pb@10ms@{SYMBOL}"


# WebSocket subscription request sent to the MEXC server after
# establishing a connection.
#
# Fields:
#   method - Operation to perform ("SUBSCRIPTION")
#   params - List of subscription topics
#   id     - Client-defined request identifier used to match responses
subscription = {
    "method": "SUBSCRIPTION",
    "params": [
        ENDPOINT
    ],
    "id": 1
}

# Lambda function that returns the current UTC timestamp as a
# formatted string with microsecond precision.
#
# Example:
#   2026-07-29 15:42:18.123456 UTC
received_time = lambda: datetime.now(
    timezone.utc
).strftime(
    "%Y-%m-%d %H:%M:%S.%f UTC"
)

# Create a synchronous WebSocket connection to the MEXC
# public market data endpoint.
#
# Returns:
#     websocket.WebSocket
ws = websocket.create_connection(
    WS_URL
)

# Serialize the subscription request dictionary into a JSON
# string and send it to the server to begin streaming data.
ws.send(
    json.dumps(subscription)
)

# Continuously receive and process incoming WebSocket messages.
# The loop runs indefinitely until the connection is closed or
# the program is terminated.
while True:

    # Block execution until the next message is received from
    # the WebSocket server.
    message = ws.recv()

    # Ignore text messages (e.g., acknowledgements, heartbeats,
    # or status responses). Only Protocol Buffer binary messages
    # are processed.
    if isinstance(message, str):
        continue

    # Create an empty Protocol Buffer wrapper object that matches
    # the structure of messages sent by the MEXC WebSocket API.
    wrapper = PushDataV3ApiWrapper()

    # Deserialize the binary Protocol Buffer payload into the
    # wrapper object, making all message fields accessible as
    # Python attributes.
    wrapper.ParseFromString(
        message
    )

    # Extract the aggregated order book depth data from the
    # decoded message.
    #
    # This object contains the current bid and ask levels
    # published by the exchange.
    orderbook = wrapper.publicAggreDepths

    # Record the local UTC time at which the message was
    # successfully received and processed.
    receive_timestamp = received_time()

    # Convert the exchange-generated timestamp (milliseconds
    # since the Unix epoch) into a timezone-aware UTC datetime.
    #
    # The sendTime field represents when the exchange generated
    # the market data message.
    exchange_timestamp = datetime.fromtimestamp(
        wrapper.sendTime / 1000,
        tz=timezone.utc
    )

   # ==============================================================================
   # BEST BID / BEST ASK
   # ==============================================================================
   #
   # Definition
   #     Best Bid = Highest buy price
   #     Best Ask = Lowest sell price
   #
   # Derivation
   #     MEXC already sorts the order book.
   #
   #     Bids : Highest → Lowest
   #     Asks : Lowest → Highest
   #
   #     Since this is a delta feed, leading price levels may have
   #     Quantity = 0 (removed orders).
   #
   #     We iterate until the first Quantity > 0.
   #
   #     Best Bid = First valid bid
   #     Best Ask = First valid ask
   #
   # Formula
   #     Spread = Best Ask − Best Bid
   #
   # Complexity
   #     Best Case : O(1)
   #     Worst Case: O(N)
   # ==============================================================================

   # ==============================================================================
   # TOP OF BOOK
   # ==============================================================================

    best_bid_price = None
    best_bid_quantity = None

    best_ask_price = None
    best_ask_quantity = None

    for bid in orderbook.bids:
        if float(bid.quantity) > 0:
            best_bid_price = float(bid.price)
            best_bid_quantity = float(bid.quantity)
            break

    for ask in orderbook.asks:
        if float(ask.quantity) > 0:
            best_ask_price = float(ask.price)
            best_ask_quantity = float(ask.quantity)
            break

    spread = None

    if best_bid_price is not None and best_ask_price is not None:
        spread = best_ask_price - best_bid_price 


   # ==============================================================================
   # MARKET SNAPSHOT
   # ==============================================================================

    print("\n" + "=" * 82)
    print("MARKET SNAPSHOT")
    print("=" * 82)

    print(f"Exchange Time : {exchange_timestamp}")
    print(f"Receive Time  : {receive_timestamp}")

    print("\n" + "=" * 82)
    print("TOP OF BOOK")
    print("=" * 82)

    if best_bid_price is not None:
        print(f"Best Bid      : {best_bid_price:.2f}")
        print(f"Bid Quantity  : {best_bid_quantity:.5f} ETH")
    else:
        print("Best Bid      : N/A")
        print("Bid Quantity  : N/A")

    print()

    if best_ask_price is not None:
        print(f"Best Ask      : {best_ask_price:.2f}")
        print(f"Ask Quantity  : {best_ask_quantity:.5f} ETH")
    else:
        print("Best Ask      : N/A")
        print("Ask Quantity  : N/A")

    print()

    if spread is not None:
        print(f"Spread        : {spread:.2f}")
    else:
        print("Spread        : N/A")

    print("=" * 82)